# Algorithm primer: Grover-based extremum finding

Walkthrough: Grover-iterate to find max/min in O(√N). SDK route: `reduce_max(x)` / `reduce_min(x)` on the simulator route runs the Grover-extremum hybrid.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the engine picks the route (classical / simulator / accelerator) at preflight time.

## Background

**Algorithm:** Grover's amplitude-amplification finds a marked element in O(√N) queries. Extending to extremum finding via reflection-based comparison gives the same quadratic speedup. SDK exposes this via `reduce_max` / `reduce_min` whose quantum lowerings emit the iterated diffusion+oracle pattern.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx import ops
from uniqx.core import types

x = np.array([3.0, 7.0, 1.0, 9.0, 4.0, 2.0, 8.0, 5.0])

@trace
def prog(arr):
    return ops.reduce_max(arr, result_type=types.scalar("f64"), axis=0)

module = prog(x.tolist())


## Preflight — see what routes the engine found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the engine's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the engine picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
